In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

## RAG 절차
- 1. docx문서를 읽는다(python-docx) : \n은 제외
- 2. 읽어온 문서를 쪼갠다(tiktoken이용)
    * 모델의 context window와 비용 확인
    * chunk(문서를 쪼갠 하나)가 길면 비용과 시간이 많이 소요
- 3. 쪼갠 문서를 임베딩 -> vector Store에 저장 => chroma(local vector Database) / pinecorn(클라우드 vector Database)
- 4. 질문과 vector DB의 유사도 검색
- 5. 유사도검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성

# 1. 문서를 읽는다(python-docx)
- pip install -U -q langchain-community python-docs docx2txt 

In [2]:
%pip install -U -q langchain-community docx2txt python-docx

Note: you may need to restart the kernel to use updated packages.


In [4]:
# 문서읽기 방법1 : 이렇게 docx파일을 읽어오면 한꺼번에 가져옴(\n포함)
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load() # \n을 포함하면 한꺼번에 load
# document
len(document)

1

In [9]:
# 문서읽기 방법2 : 문단씩 가져오기
from docx import Document
documents = Document('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
print(documents)
print(dir(documents))

['_Document__body', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_width', '_body', '_element', '_parent', '_part', 'add_comment', 'add_heading', 'add_page_break', 'add_paragraph', 'add_picture', 'add_section', 'add_table', 'comments', 'core_properties', 'element', 'inline_shapes', 'iter_inner_content', 'paragraphs', 'part', 'save', 'sections', 'settings', 'styles', 'tables']


In [17]:
len(documents.paragraphs) # \n마다 새로운 paragraphs로 생성
for paragraphs in documents.paragraphs[:5]:
    # print(dir(paragraphs))
    print('-------------------')
    print(paragraphs.text)

-------------------

-------------------
소득세법
-------------------
[시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정]
-------------------
재정경제부(재산세제과(양도소득세)) 044-215-4312
-------------------
재정경제부(소득세제과(근로소득)) 044-215-4216


In [20]:
# \n이 제거된 문서 전체 내용을 full_text
full_text = ""
for paragraph in documents.paragraphs:
    full_text += paragraph.text + " "
full_text[:100]

' 소득세법 [시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정] 재정경제부(재산세제과(양도소득세)) 044-215-4312 재정경제부(소득세제'

# 2. 문서를 쪼갠다
- pip install tiktoken : openAI의 공식 토크나이저 라이브러리
    - full_text -> 토큰단위로 쪼개서 숫자
    - 숫자 -> 문자
    - 1500(?)토큰씩 문서를 쪼개기

In [22]:
%pip install -q tiktoken

Note: you may need to restart the kernel to use updated packages.


In [29]:
import tiktoken
encoder = tiktoken.encoding_for_model('gpt-4o-mini')
# full_text(문자들) -> 숫자리스트
encoding = encoder.encode(full_text)
# 숫자리스트 -> 문자들
decoded = encoder.decode(encoding)

In [34]:
print(f"글자수 : {len(full_text)}")
print(f"토큰수 : {len(encoding)}")
print('문자들 :', decoded[:10])
print('숫자들 :', encoding[:10])

글자수 : 248301
토큰수 : 162419
문자들 :  소득세법 [시행 
숫자들 : [22183, 64328, 11734, 22070, 723, 5637, 15719, 220, 1323, 21]


In [39]:
print(encoder.encode('Hello'), encoder.encode('안녕'), encoder.encode('친구'))

[13225] [14307, 98931] [61407, 10997]


In [41]:
print(list(range(0, 162419, 1500)))

[0, 1500, 3000, 4500, 6000, 7500, 9000, 10500, 12000, 13500, 15000, 16500, 18000, 19500, 21000, 22500, 24000, 25500, 27000, 28500, 30000, 31500, 33000, 34500, 36000, 37500, 39000, 40500, 42000, 43500, 45000, 46500, 48000, 49500, 51000, 52500, 54000, 55500, 57000, 58500, 60000, 61500, 63000, 64500, 66000, 67500, 69000, 70500, 72000, 73500, 75000, 76500, 78000, 79500, 81000, 82500, 84000, 85500, 87000, 88500, 90000, 91500, 93000, 94500, 96000, 97500, 99000, 100500, 102000, 103500, 105000, 106500, 108000, 109500, 111000, 112500, 114000, 115500, 117000, 118500, 120000, 121500, 123000, 124500, 126000, 127500, 129000, 130500, 132000, 133500, 135000, 136500, 138000, 139500, 141000, 142500, 144000, 145500, 147000, 148500, 150000, 151500, 153000, 154500, 156000, 157500, 159000, 160500, 162000]


In [54]:
import tiktoken
def split_text(full_text=full_text, chunk_size=1500):
    'full_text를 chunk_size 토큰만큼 쪼개서 chunk_list로 반환'
    encoder = tiktoken.encoding_for_model('gpt-4o-mini') # nano모델은 안 됨
    total_encoding = encoder.encode(full_text) # full_text를 토큰단위 숫자 list
    total_token_count = len(total_encoding) # full_text의 토큰 수
    chunk_list = []
    for i in range(0, total_token_count, chunk_size):
        chunk_encoding = total_encoding[i : i+chunk_size]
        chunk_encoded = encoder.decode(chunk_encoding)
        #print(chunk_encoding,'=>', chunk_encoded)
        chunk_list.append(chunk_encoded)
    return chunk_list

In [55]:
example_text = "소득세법 법률 일부 개정함 이자소득 배당소득 무시기 무시기"
split_text(full_text=example_text, chunk_size=10)

['소득세법 법률 일부 개정함', ' 이자소득 배당소득 무시', '기 무시기']

In [61]:
chunk_list = split_text(full_text, 1500)
len(chunk_list)

109

# 3. 쪼갠 문서를 임베딩 -> vector Store(=vector Database, 인메모리)에 저장
- pip install chromadb

In [64]:
%pip install -q chromadb --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import chromadb
chroma_client = chromadb.Client()